In [1]:
from vllm import LLM, SamplingParams
import pandas as pd
import numpy as np

def get_perplexity(token_list):
    """
        Calcula perplexity para una instancia única.
    """
    t = len(token_list)
    avg = (-1/t) * np.sum(token_list)
    perplexity = np.exp(avg)
    return round(perplexity, 4)

train = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl'
dataset = pd.read_json(train, lines = True)

trans_prompt = """
    Given a set of premises, the task is to parse the problem and the question into first-order logic formulars. Answer only with the translated premises.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Natural Language Premises:
    All people who regularly drink coffee are dependent on caffeine. People either regularly drink coffee or joke about being addicted to caffeine. No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    Predicates:
    Dependent(x) ::: x is a person dependent on caffeine.
    Drinks(x) ::: x regularly drinks coffee.
    Jokes(x) ::: x jokes about being addicted to caffeine.
    Unaware(x) ::: x is unaware that caffeine is a drug.
    Student(x) ::: x is a student.
    FOL Premises:
    ∀x (Drinks(x) → Dependent(x)) ::: All people who regularly drink coffee are dependent on caffeine.
    ∀x (Drinks(x) ⊕ Jokes(x)) ::: People either regularly drink coffee or joke about being addicted to caffeine.
    ∀x (Jokes(x) → ¬Unaware(x)) ::: No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. 
    (Student(rina) ∧ Unaware(rina)) ⊕ ¬(Student(rina) ∨ Unaware(rina)) ::: Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. 
    ¬(Dependent(rina) ∧ Student(rina)) → (Dependent(rina) ∧ Student(rina)) ⊕ ¬(Dependent(rina) ∨ Student(rina)) ::: If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    --------------
    
    Natural Language Premises:
    {}
    FOL Premises:
"""

infer_prompt = """
    Given a set of premises and conclusion in first order logic, your task is to determine the logical validity of the conclusion: True, False, or Uncertain. Answer only with the logical value.
    A True conclusion is one that can be obtained via a valid inference procedure from the given premises.
    A False conclusion is one that contradicts one or more premises during the inference procedure. 
    An Uncertain conclusion is neither True nor False. Meaning that there is insufficient information in the premises to infer it, but the conclusion it self doesn't contradict any premise.
    --------------
    The following example shows a set of premises and conclusions where each conclusion represents a different logical validity. You should answer similarly.
    FOL-PREMISES:
    ∀x (WorkAt(x, meta) → HighIncome(x))
    ∀x (HighIncome(x) → ¬MeansToDestination(x, bus))
    ∀x (MeansToDestination(x, bus) ⊕ MeansToDestination(x, drive))
    ∀x (HaveCar(x) → MeansToDestination(x, drive))
    ∀x (Student(x) → ¬ MeansToDestination(x, drive))
    HaveCar(james) ∨ WorkAt(james, meta)
    --------------
    FOL-CONCLUSION:
    MeansToDestination(x, drive) ∨ Student(james)
    Student(james)
    ¬HighIncome(james)

    Analysis:
    The first conclusion is True. Premise 6 states that either James has a car (in which case premise 4 gives us the conclusion) or James works at Meta (in which case premise 4 implies premise 2, which combined with premise 3 gives us the conclusion)
    The second conclusion is False. Premise 5 states that students can't have a Car as a MeansToDestination, however the first condition tells us James has such means.
    The third conclusion is Uncertain. Premise 1 is the only guarantee to have a High Income, however we can't determine that James works at Meta (Premise 6).
    ----------------------------
    FOL-PREMISES:
    {}
    --------------
    FOL-CONCLUSION:
    {}
    --------------
    ANSWER:
"""


retrans_prompt = """
    Given a single premise in first order logic, your task is to translate the premise into natural language. Answer only with the translated premise. It should be a single sentence.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Below are examples of the translation:
    PREMISES:
    ¬(PartTime(jackie) ⊕ ForbesList(jackie)) → ∃y (LessThan(y, num2) ∧ TakesCourses(x,y)) ∧ ForbesList(jackie)
    ¬In(borjMasouda, tunisia)

    NATURAL LANGUAGE:
    If Jackie either enrolls as part-time in the current semester and is listed in the Forbes 30 Under 30, or neither enrolls as part-time in the current semester nor is listed in the Forbes 30 Under 30, then Jackie takes less than two courses in the current semester and listed in the Forbes 30 Under 30.
    Borj Masouda is not in Tunisia.
    --------------    
    PREMISE:
    {}

    NATURAL LANGUAGE:
"""

In [2]:
def vllm_generation(model_id, quant):
    """
        Generación iterativa sobre distintos modelos usando vLLM.
    """
    print(f"Iniciando con: {model_id}...")
    
    trans_prompts = [trans_prompt.format(dataset['premises'][i]) for i in range(len(dataset['premises']))]
    infer_prompts = [infer_prompt.format(dataset['premises-FOL'][i], dataset['conclusion-FOL'][i]) for i in range(len(dataset['premises-FOL']))]
    retrans_prompts = [retrans_prompt.format(dataset['conclusion-FOL'][i]) for i in range(len(dataset['conclusion-FOL']))]

    prompts = trans_prompts + infer_prompts + retrans_prompts

    sampling_params = SamplingParams(temperature = 0.2, top_p = 0.9, max_tokens = 4000, seed = 67, logprobs = 1)
    llm = LLM(
        model = model_id, 
        max_model_len = 6000, 
        quantization = quant, 
        enforce_eager = True, 
        gpu_memory_utilization = 0.9, 
        limit_mm_per_prompt={"image": 0, "video": 0}
    )
    outputs = llm.generate(prompts, sampling_params)

    step_list = []
    perp_list = []
    i = 0
    for output in outputs:
        gen_text = output.outputs[0].text.strip(r'\n')
        step_list.append(gen_text)
        loglist = output.outputs[0].logprobs
        logit_values = [loglist[i].get(list(loglist[i].keys())[0]).logprob for i in range(len(loglist))]
        perp_list.append(get_perplexity(logit_values))
        if i % 20 == 0:
            print('Iteración: {}'.format(i))
        i += 1

        
    # To do: Pensar en otras cosas que medirle a las generaciones de los modelos.
    df_name = model_id.split('/')[1] # AHHHHHHHHHHHHHHHHHHHHHHH
    full_dataframe = pd.DataFrame({f"Full": step_list, "Perplexity": perp_list})
    full_dataframe.to_csv(f'/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{df_name}_full.csv')
    print("=" * 60)
    print("=" * 15 +  f" Fin de: {model_id} " + "=" * 15)
    print("=" * 60)


# 27B-32B no funciona papus :v Se queda al límite de la memoria. Usa 31.28GBs y Necesita 31.6GBs. Hay que jugarle con gpu_offloading, pero será para otro paper.
checkpoint_list = [
    #['Qwen/Qwen3-4B-FP8', 'fp8'], # Funciona bien. DONE
    #['Qwen/Qwen3-8B-FP8', 'fp8'], # Funciona bien. DONE
    #['Qwen/Qwen3-14B-FP8', 'fp8'], # Fuinciona bien. DONE
    #['Qwen/Qwen3.5-4B', 'fp8'], # Funciona bien. DONE # Estamos haciendo pruebas con max_model_len = 7000, y max_tokens = 5000
    #['Qwen/Qwen3.5-9B', 'fp8'], # Funciona bien. DONE
    #['google/gemma-3-4b-it','fp8'], # Funciona bien. DONE
    #['google/gemma-3-12b-it','fp8'], # Funciona bien. DONE
    #['openai/gpt-oss-20b', 'mxfp4'], # Funciona bien. DONE
    #['deepseek-ai/DeepSeek-R1-Distill-Llama-8B', None], # Jala bien.
    ['deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', None], # Jala bien.
    ['deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', None],
    ['deepseek-ai/DeepSeek-R1-0528-Qwen3-8B', None]
]

# checkpoints que no corren: 
#[
# ['deepseek-ai/DeepSeek-R1-Distill-Qwen-14B', None], #Jala bien pero con gpu_memory_utilization = 0.95 # TENEMOS QUE OPTIMIZAR ESTO PTM.
# ['Qwen/Qwen3.5-27B-FP8', 'fp8'], 
# ['google/gemma-4-E2B', 'fp8'], 
# ['google/gemma-4-E4B', 'fp8'],
# ['google/gemma-4-26B-A4B','fp8'],
# ['google/gemma-3-27b-it','fp8'], 
#]

for _ in checkpoint_list:
    vllm_generation(_[0], _[1])
#vllm_generation(checkpoint_list[0][0], checkpoint_list[0][1])

Iniciando con: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B...
INFO 04-13 15:23:36 [utils.py:233] non-default args: {'max_model_len': 6000, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'}


KeyboardInterrupt: 